In [1]:
import os
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.utils import shuffle
from scipy.signal import resample
import matplotlib.pyplot as plt
import json
import warnings
import mne
from mne.preprocessing import ICA
warnings.filterwarnings("ignore")
try:
    from mne_icalabel import label_components
except Exception:
    label_components = None

In [21]:
SAMPLE_RATE = 200  # fs
# baseline correction according to paper, tmin and tmax was observed from the data
TMIN = -0.2
TMAX = 0.8
BASELINE=(-0.2, 0.0)
TASK_ID = 1  # Visual oddball
TASK_NAME = 'oddball'
DATASET_NAME = 'TDBrain-ODD'
MONTAGE = 'standard_1020'
sub_folder_path = str(SAMPLE_RATE) + 'Hz'
sub_folder_path

'200Hz'

In [10]:
# root dir
root = 'TDBRAIN_Dataset_V3_1'
# participants file path
ptc_path = os.path.join(root, 'TDBRAIN_participants_V3.xlsx')
ptc = pd.read_excel(ptc_path)
ptc

,TDBRAIN_ID,DISC/REP,indication,formal_status,Dataset,Consent,sessSeason,sessTime,Responder,Remitter,...,BDI_pre,BDI_post,rTMS PROTOCOL,ADHD_pre_Hyp_leading,ADHD_pre_Att_leading,ADHD_post_Att_leading,ADHD_post_Hyp_leading,NF Protocol,YBOCS_pre,YBOCS_post
0,sub-87957837,DISCOVERY,HEALTHY,HEALTHY,NaN,YES,summer,afternoon,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,sub-87957881,DISCOVERY,HEALTHY,HEALTHY,NaN,YES,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,sub-87957969,DISCOVERY,HEALTHY,HEALTHY,NaN,YES,spring,morning,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,sub-87958013,DISCOVERY,HEALTHY,HEALTHY,NaN,YES,summer,afternoon,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,sub-87958057,DISCOVERY,HEALTHY,HEALTHY,NaN,YES,summer,afternoon,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1358,sub-88078657,DISCOVERY,INSOMNIA,INSOMNIA,NaN,YES,fall,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1359,sub-88078749,DISCOVERY,INSOMNIA,INSOMNIA,NaN,YES,winter,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1360,sub-88078793,DISCOVERY,INSOMNIA,INSOMNIA,NaN,YES,winter,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1361,sub-88078837,DISCOVERY,INSOMNIA,INSOMNIA,NaN,YES,winter,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
# EEG data preprocessing

# all disease mapping
all_dise_dict = {}
ls_ind = list(ptc['indication'].unique())
for i in range(len(ls_ind)):
    all_dise_dict[ls_ind[i]] = i
print(all_dise_dict)
    
ptc['indication_id'] = ptc['indication'].map(all_dise_dict)
ptc

{'HEALTHY': 0, 'BURNOUT': 1, 'SMC': 2, 'Dyslexia': 3, 'CHRONIC PAIN': 4, 'MDD': 5, nan: 6, 'ADHD': 7, 'ADHD/ASPERGER': 8, 'PDD NOS/DYSLEXIA': 9, 'PDD NOS': 10, 'WHIPLASH': 11, 'ANXIETY': 12, 'ADHD/DYSLEXIA': 13, 'ASD': 14, 'TINNITUS': 15, 'OCD': 16, 'Tinnitus': 17, 'PDD NOS ': 18, 'PANIC': 19, 'MDD/ANXIETY': 20, 'MIGRAINE': 21, 'PDD NOS/ANXIETY': 22, 'PARKINSON': 23, 'BIPOLAR': 24, 'MDD/bipolar': 25, 'DYSPRAXIA': 26, 'TINNITUS/MDD': 27, 'ADHD/ASD/ANXIETY': 28, 'MDD/ADHD': 29, 'ADHD/PDD NOS': 30, 'MDD/BIPOLAR': 31, 'ASPERGER': 32, 'ADHD/EPILEPSY': 33, 'MDD/PAIN': 34, 'PDD NOS/GTS': 35, 'PDD NOS/ADHD': 36, 'PDD NOS/ASD': 37, 'TBI': 38, 'ADHD/ANXIETY': 39, 'ADHD/DYSLEXIA/DYSCALCULIA': 40, 'ADHD/MDD': 41, 'MDD/PANIC': 42, 'DEPERSONALIZATION': 43, 'MDD/TRAUMA': 44, 'PTSD/ADHD': 45, 'OCD/DPS': 46, 'MDD/OCD': 47, 'MDD/TUMOR': 48, 'ADHD/GTS': 49, 'OCD/MDD': 50, 'CONVERSION DX': 51, 'ASD/ASPERGER': 52, 'MDD/ADHD/LYME': 53, 'ADHD/OCD': 54, 'MSA-C': 55, 'OCD/ASD': 56, 'STROKE/PAIN': 57, 'STROKE '

,TDBRAIN_ID,DISC/REP,indication,formal_status,Dataset,Consent,sessSeason,sessTime,Responder,Remitter,...,BDI_post,rTMS PROTOCOL,ADHD_pre_Hyp_leading,ADHD_pre_Att_leading,ADHD_post_Att_leading,ADHD_post_Hyp_leading,NF Protocol,YBOCS_pre,YBOCS_post,indication_id
0,sub-87957837,DISCOVERY,HEALTHY,HEALTHY,NaN,YES,summer,afternoon,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,sub-87957881,DISCOVERY,HEALTHY,HEALTHY,NaN,YES,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,sub-87957969,DISCOVERY,HEALTHY,HEALTHY,NaN,YES,spring,morning,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,sub-87958013,DISCOVERY,HEALTHY,HEALTHY,NaN,YES,summer,afternoon,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,sub-87958057,DISCOVERY,HEALTHY,HEALTHY,NaN,YES,summer,afternoon,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1358,sub-88078657,DISCOVERY,INSOMNIA,INSOMNIA,NaN,YES,fall,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62
1359,sub-88078749,DISCOVERY,INSOMNIA,INSOMNIA,NaN,YES,winter,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62
1360,sub-88078793,DISCOVERY,INSOMNIA,INSOMNIA,NaN,YES,winter,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62
1361,sub-88078837,DISCOVERY,INSOMNIA,INSOMNIA,NaN,YES,winter,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62


In [17]:
# Test for bad channels, sampling freq and shape
bad_channel_list, sampling_freq_list, data_shape_list = [], [], []

for sub in os.listdir(root):
    if 'sub-' in sub:
        sub_path = os.path.join(root, sub, 'eeg/')
        for ses in os.listdir(os.path.join(root, sub)):
            if 'ses-' in ses:
                eeg_path = os.path.join(root, sub, ses, 'eeg/')
                for file in os.listdir(eeg_path):
                    if '.bdf' in file and f'task-{TASK_NAME}' in file:
                        file_path = os.path.join(eeg_path, file)
                        print(file_path)
                        try:
                            raw = mne.io.read_raw_bdf(file_path, preload=True)
                            # get bad channels
                            bad_channel = raw.info['bads']
                            bad_channel_list.append(bad_channel)
                            # get sampling frequency
                            sampling_freq = raw.info['sfreq']
                            sampling_freq_list.append(sampling_freq)
        
                            # get eeg data
                            data = raw.get_data()
                            data_shape = data.shape
                            data_shape_list.append(data_shape)
                        except Exception as e:
                            print(f"Error processing {file_path}: {e}")

TDBRAIN_Dataset_V3_1\sub-87957837\ses-1\eeg/sub-87957837_ses-1_task-oddball_eeg.bdf
Extracting EDF parameters from D:\PycharmPorjects\DataProcessingLocalDisk\ERP-FM\TDBrain-ODD\TDBRAIN_Dataset_V3_1\sub-87957837\ses-1\eeg\sub-87957837_ses-1_task-oddball_eeg.bdf...
BDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 180499  =      0.000 ...   360.998 secs...
TDBRAIN_Dataset_V3_1\sub-87957881\ses-1\eeg/sub-87957881_ses-1_task-oddball_eeg.bdf
Extracting EDF parameters from D:\PycharmPorjects\DataProcessingLocalDisk\ERP-FM\TDBrain-ODD\TDBRAIN_Dataset_V3_1\sub-87957881\ses-1\eeg\sub-87957881_ses-1_task-oddball_eeg.bdf...
BDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 180499  =      0.000 ...   360.998 secs...
TDBRAIN_Dataset_V3_1\sub-87957969\ses-1\eeg/sub-87957969_ses-1_task-oddball_eeg.bdf
Extracting EDF parameters from D:\PycharmPorjects\DataProcessingLocalDisk\ERP-FM\TDBrain-ODD\TDBRAIN_Dataset

In [18]:
from collections import Counter

print(bad_channel_list)
print(data_shape_list[0])
print("Channel number counter:", Counter(i[0] for i in data_shape_list))
print("Sampling rate counter:", Counter(sampling_freq_list))

[[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []]
(33, 180500)
Channel number counter: Counter({33: 162})
Sampling rate counter: Counter({500.0: 162})


In [19]:
common_channels = None
for sub in os.listdir(root):
    if 'sub-' in sub:
        sub_path = os.path.join(root, sub, 'eeg/')
        for ses in os.listdir(os.path.join(root, sub)):
            if 'ses-' in ses:
                eeg_path = os.path.join(root, sub, ses, 'eeg/')
                for file in os.listdir(eeg_path):
                    if '.bdf' in file and f'task-{TASK_NAME}' in file:
                        file_path = os.path.join(eeg_path, file)
                        print(file_path)
                        try:
                            raw = mne.io.read_raw_bdf(file_path, preload=True)
                            current_channels = set(raw.info['ch_names'])
                            if not common_channels:
                                common_channels = current_channels
                            else:
                                common_channels &= current_channels
                        except Exception as e:
                            print(f"Error processing {file_path}: {e}")
                            continue
                            
common_channels = list(common_channels)
print(f"Common channels ({len(common_channels)}):", common_channels)

TDBRAIN_Dataset_V3_1\sub-87957837\ses-1\eeg/sub-87957837_ses-1_task-oddball_eeg.bdf
Extracting EDF parameters from D:\PycharmPorjects\DataProcessingLocalDisk\ERP-FM\TDBrain-ODD\TDBRAIN_Dataset_V3_1\sub-87957837\ses-1\eeg\sub-87957837_ses-1_task-oddball_eeg.bdf...
BDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 180499  =      0.000 ...   360.998 secs...
TDBRAIN_Dataset_V3_1\sub-87957881\ses-1\eeg/sub-87957881_ses-1_task-oddball_eeg.bdf
Extracting EDF parameters from D:\PycharmPorjects\DataProcessingLocalDisk\ERP-FM\TDBrain-ODD\TDBRAIN_Dataset_V3_1\sub-87957881\ses-1\eeg\sub-87957881_ses-1_task-oddball_eeg.bdf...
BDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 180499  =      0.000 ...   360.998 secs...
TDBRAIN_Dataset_V3_1\sub-87957969\ses-1\eeg/sub-87957969_ses-1_task-oddball_eeg.bdf
Extracting EDF parameters from D:\PycharmPorjects\DataProcessingLocalDisk\ERP-FM\TDBrain-ODD\TDBRAIN_Dataset

In [20]:
# remove non-eeg electrodes
non_eeg_channels = [
    'Erbs', 'Status', 'HNHR', 'Mass', 'VPVA', 'HPHL', 'VNVB'
]
common_channels = [ch for ch in common_channels if ch not in non_eeg_channels]
print(common_channels)
print("Common channels number: ", len(common_channels))

['Fp1', 'Fp2', 'CP4', 'Pz', 'Cz', 'CP3', 'P8', 'P7', 'Fz', 'P3', 'FCz', 'Oz', 'C4', 'T8', 'FC3', 'F3', 'O2', 'F7', 'P4', 'O1', 'F8', 'FC4', 'CPz', 'T7', 'F4', 'C3']
Common channels number:  26


In [22]:
def data_preprocessing(
        raw: mne.io.Raw,
        common_channels: list,
        sample_rate: int = 250,
        notch_freq: float = 60.0,
        l_freq: float = 0.5,
        h_freq: float = 40.0,
        do_bad_interp: bool = True,
        verbose: bool = True,
):
    """
    Preprocessing steps ：
      1) choose common channels and reorder
      2) Set Montage
      3) 60 Hz Notch（before band pass）
      4) bandpass filter（default 0.5–40 Hz）
      5) interpolate bad channels（if do_bad_interp is True）
      6) re-reference to average
      7) ICA（在 1 Hz 高通的副本上拟合，自动剔除眼动/肌电等分量，需 mne-icalabel）
      8) downsample to 250 Hz
    """

    # 1. select common channels and reorder to given order
    print("Current channels: ", raw.ch_names)
    keep = [ch for ch in common_channels if ch in raw.ch_names]
    raw.pick_channels(keep)
    raw.reorder_channels(keep)
    if verbose:
        print(f"✔ Step 2: Picked common EEG channels ({len(keep)}): {keep}")

    # 2. Set Montage
    raw.set_montage(mne.channels.make_standard_montage(MONTAGE))
    if verbose:
        print(f"✔ Step 1, Montage set: {MONTAGE}.")

    # 3. Notch（工频）
    if notch_freq is not None:
        raw.notch_filter(freqs=[notch_freq], picks="eeg", verbose=False)
        if verbose:
            print(f"✔ Step 3: Notch @ {notch_freq} Hz")

    # 4. Bandpass Filter (0.5–40 Hz)
    raw.filter(l_freq=l_freq, h_freq=h_freq, picks="eeg", verbose=False)
    if verbose:
        print(f"✔ Step 4: Band-pass {l_freq}–{h_freq} Hz")

    # 5. Interpolate bad channels
    if do_bad_interp and raw.info.get("bads"):
        raw.interpolate_bads(reset_bads=True, verbose=False)
        if verbose:
            print(f"✔ Step 5: Interpolated bads: {raw.info.get('bads', [])}")
    else:
        if verbose:
            print("ℹ Step 5: No bads to interpolate (set raw.info['bads'] first if needed)")

    # 6) Re-reference to average
    raw.set_eeg_reference("average", verbose=False)
    if verbose:
        print("✔ Step 6: Average reference")
        print()

    # 7) ICA (fit ICLabel on 1 Hz high-pass filtered copy, then apply to original)
    raw_for_ica = raw.copy().filter(l_freq=1.0, h_freq=None, picks="eeg", verbose=False)

    ica = ICA(n_components=None, method="fastica", random_state=97, max_iter="auto")
    ica.fit(raw_for_ica)

    excluded = []
    if label_components is not None:
        try:
            ic_labels = label_components(raw_for_ica, ica, method="iclabel")
            labels = ic_labels["labels"]
            probs = ic_labels["y_pred_proba"]  # (n_comp, n_classes)

            thresholds = {
                "eye blink": 0.7,
                "muscle artifact": 0.6,
                "heart beat": 0.5,
                "line noise": 0.8,
                "channel noise": 0.9,
            }

            for i, lab in enumerate(labels):
                if lab in thresholds:
                    if probs is not None:
                        p = probs[i].max()
                    else:
                        p = 1.0
                    if p >= thresholds[lab]:
                        excluded.append(i)

        except Exception as e:
            if verbose:
                print(f"⚠ ICLabel failed ({e}). Skipping auto exclusion.")
    else:
        if verbose:
            print("ℹ ICLabel not available; fitted ICA but no auto component exclusion.")

    if excluded:
        ica.exclude = sorted(set(excluded))
        raw = ica.apply(raw.copy())
        if verbose:
            print(f"✔ Step 7: ICA applied. Excluded comps: {ica.exclude}")
    else:
        if verbose:
            print("ℹ Step 7: No ICA components excluded.")

    # 8) downsample to 250 Hz
    if raw.info["sfreq"] != sample_rate:
        raw.resample(sample_rate, npad="auto", verbose=False)
        if verbose:
            print(f"✔ Step 8: Resampled to {sample_rate} Hz")

    return raw

In [23]:
def epoch_and_make_xy(
        raw: mne.io.Raw,
        events_tsv_path: str,
        tmin: float = -0.2,
        tmax: float = 0.8,
        baseline=(-0.2, 0),
        task_id: int = 1,
        subject_id: int = 1,
        disease_id: int = -1,
):
    """
    Extract standard/target stimulus-locked epochs.

    Output:
        X: (n_trials, n_times, n_channels)
        y: (n_trials, 4) = [disease_id, stimulus_type, subject_id, task_id]

    stimulus_type:
        0 = standard
        1 = target
    """

    ev = pd.read_csv(events_tsv_path, sep="\t").copy()

    required_cols = ["onset", "trial_type"]
    for col in required_cols:
        if col not in ev.columns:
            raise ValueError(
                f"Column '{col}' not found in {events_tsv_path}. "
                f"Available columns: {list(ev.columns)}"
            )

    ev["trial_type"] = ev["trial_type"].astype(str).str.strip().str.lower()

    if "stim_type" in ev.columns:
        ev["stim_type"] = ev["stim_type"].astype(str).str.strip().str.lower()
        mismatch = (
            ev["trial_type"].isin(["standard", "target"]) &
            ev["stim_type"].isin(["standard", "target"]) &
            (ev["trial_type"] != ev["stim_type"])
        )
        if mismatch.any():
            print(
                f"Warning: trial_type and stim_type mismatch in "
                f"{mismatch.sum()} rows. Using trial_type as label."
            )

    label_map = {
        "standard": 0,
        "target": 1,
    }

    label_names = {
        0: "standard",
        1: "target",
    }

    stim_ev = ev[ev["trial_type"].isin(label_map.keys())].copy()
    stim_ev = stim_ev.dropna(subset=["onset"]).copy()

    if len(stim_ev) == 0:
        print("Available trial_type values:")
        print(ev["trial_type"].value_counts(dropna=False))
        raise ValueError(
            f"No valid standard/target events found in {events_tsv_path}."
        )

    stim_ev["stimulus_type"] = (
        stim_ev["trial_type"]
        .map(label_map)
        .astype(int)
    )

    sfreq = raw.info["sfreq"]
    print(f"Current sampling frequency: {sfreq} Hz")

    # onset is in seconds. Do not use sample column directly,
    # because raw may have been resampled.
    samples = np.round(
        stim_ev["onset"].values.astype(float) * sfreq
    ).astype(int)

    events = np.c_[
        samples,
        np.zeros_like(samples, dtype=int),
        np.ones_like(samples, dtype=int)
    ]

    picks = mne.pick_types(
        raw.info,
        eeg=True,
        eog=False,
        exclude="bads"
    )

    epochs = mne.Epochs(
        raw,
        events,
        event_id={"stimulus": 1},
        tmin=tmin,
        tmax=tmax,
        baseline=baseline,
        picks=picks,
        proj=False,
        preload=True,
        reject=None,
        verbose=False,
    )

    data = epochs.get_data()  # (N, C, T)
    target_len = int(round((tmax - tmin) * sfreq))

    if data.shape[-1] > target_len:
        data = data[..., :target_len]

    elif data.shape[-1] < target_len:
        pad = target_len - data.shape[-1]
        data = np.pad(
            data,
            ((0, 0), (0, 0), (0, pad)),
            mode="edge"
        )

    X = np.transpose(data, (0, 2, 1))  # (N, T, C)

    # Align labels if MNE drops out-of-bound epochs.
    trial_type = stim_ev["stimulus_type"].values.astype(int)
    trial_type = trial_type[epochs.selection]

    y = np.column_stack([
        np.full_like(trial_type, disease_id),  # disease_id
        trial_type,                            # stimulus_type
        np.full_like(trial_type, subject_id),  # subject_id
        np.full_like(trial_type, task_id),     # task_id
    ])

    assert X.shape[0] == y.shape[0], (
        f"X/y mismatch: X={X.shape}, y={y.shape}"
    )

    print("Task: standard-target stimulus classification")
    print(f"Number of retained trials: {len(trial_type)}")
    print("Label distribution:")

    labels, counts = np.unique(trial_type, return_counts=True)

    for label, count in zip(labels, counts):
        print(f"  {label}: {label_names[label]} = {count}")

    return X, y

In [24]:
import re

def natural_sort_key(text):
    """
    Sort run names naturally:
        run-1, run-2, ..., run-10
    instead of:
        run-1, run-10, run-2
    """
    return [
        int(part) if part.isdigit() else part.lower()
        for part in re.split(r"(\d+)", text)
    ]

## PASS 1: Find total number of trials across all subjects

In [28]:
N_total = 0  # total number of trials across all subjects, sessions and runs

for sub in sorted(os.listdir(root), key=natural_sort_key):
    if not sub.startswith("sub-"):
        continue

    print(f"Current subject: {sub}")
    sub_path = os.path.join(root, sub)

    if not os.path.isdir(sub_path):
        continue

    subject_id = int(sub.split("-")[-1])
    print(f"Subject ID: {subject_id}")

    # Iterate over sessions
    for ses in sorted(os.listdir(sub_path), key=natural_sort_key):
        if not ses.startswith("ses-"):
            continue

        print(f"  Current session: {ses}")
        eeg_path = os.path.join(sub_path, ses, "eeg")

        if not os.path.exists(eeg_path):
            print(f"❌ Skipping {sub}/{ses}: no eeg folder found.")
            continue

        set_files = {}
        events_files = {}

        for filename in os.listdir(eeg_path):
            filepath = os.path.join(eeg_path, filename)

            if filename.endswith("_eeg.bdf"):
                run_prefix = filename[:-len("_eeg.bdf")]
                set_files[run_prefix] = filepath

            elif filename.endswith("_events.tsv"):
                run_prefix = filename[:-len("_events.tsv")]
                events_files[run_prefix] = filepath

        all_run_prefixes = sorted(
            set(set_files.keys()) | set(events_files.keys()),
            key=natural_sort_key
        )

        # Only keep TASK_NAME runs
        all_run_prefixes = [
            run_prefix for run_prefix in all_run_prefixes
            if TASK_NAME in run_prefix
        ]
        
        if len(all_run_prefixes) == 0:
            print(f"❌ Skipping {sub}/{ses}: no run files found.")
            continue

        print(f"  Number of detected runs: {len(all_run_prefixes)}")

        for run_prefix in all_run_prefixes:
            set_file_path = set_files.get(run_prefix)
            events_file_path = events_files.get(run_prefix)

            print(f"    Current run: {run_prefix}")

            if set_file_path is None:
                print(f"    ❌ Missing .set file for {run_prefix}, skipping.")
                continue

            if events_file_path is None:
                print(f"    ❌ Missing events.tsv file for {run_prefix}, skipping.")
                continue

            print("    Start loading EEG data...")
            raw = mne.io.read_raw_bdf(
                set_file_path,
                preload=True,
                verbose=False
            )

            print(f"Subject ID: {subject_id}")
            print("Start epoching and making X, y...")
    
            # Remove beh_file_path, keep other parameters unchanged
            X, y = epoch_and_make_xy(
                raw, events_file_path,
                tmin=TMIN, tmax=TMAX, baseline=BASELINE,
                task_id=TASK_ID, subject_id=subject_id,
            )
    
            print(f"X shape: {X.shape}, y shape: {y.shape}")  
            # X: (N, SAMPLE_RATE * (tmax - tmin), C)
            # y: (N, 4) = [disease_id, stimulus_type, subject_id, task_id]
    
            N_total += X.shape[0]
    
            print("------------------------------------------------\n")

        print("  ================================================\n")

print(f"Total number of trials across all subjects, sessions and runs: {N_total}")

Current subject: sub-87957837
Subject ID: 87957837
  Current session: ses-1
  Number of detected runs: 1
    Current run: sub-87957837_ses-1_task-oddball
    Start loading EEG data...
Subject ID: 87957837
Start epoching and making X, y...
Current sampling frequency: 500.0 Hz
Task: standard-target stimulus classification
Number of retained trials: 340
Label distribution:
  0: standard = 280
  1: target = 60
X shape: (340, 500, 32), y shape: (340, 4)
------------------------------------------------


Current subject: sub-87957881
Subject ID: 87957881
  Current session: ses-1
  Number of detected runs: 1
    Current run: sub-87957881_ses-1_task-oddball
    ❌ Missing events.tsv file for sub-87957881_ses-1_task-oddball, skipping.

Current subject: sub-87957969
Subject ID: 87957969
  Current session: ses-1
  Number of detected runs: 1
    Current run: sub-87957969_ses-1_task-oddball
    Start loading EEG data...
Subject ID: 87957969
Start epoching and making X, y...
Current sampling frequenc

## Create memmap storage

In [34]:
output_root = os.path.join('Processed', sub_folder_path, DATASET_NAME)
os.makedirs(output_root, exist_ok=True)

X_path = os.path.join(output_root, 'X.dat')
y_path = os.path.join(output_root, 'y.dat')
meta_path = os.path.join(output_root, 'meta.json')

print("X path:", X_path)
print("y path:", y_path)

# create memmap storage
SEG_LEN = int((TMAX - TMIN) * SAMPLE_RATE)  # samples per epoch
n_channels = len(common_channels)
X_mm = np.memmap(X_path, dtype='float32', mode='w+', shape=(N_total, SEG_LEN, n_channels))
y_mm = np.memmap(y_path, dtype='float32', mode='w+', shape=(N_total, 4))
print("Created memmap for X with shape:", X_mm.shape)

X path: Processed\200Hz\TDBrain-ODD\X.dat
y path: Processed\200Hz\TDBrain-ODD\y.dat
Created memmap for X with shape: (54400, 200, 26)


## PASS 2: Process and store trials into memmap

In [35]:
cur = 0  # current index in memmap

for sub in sorted(os.listdir(root), key=natural_sort_key):
    if not sub.startswith("sub-"):
        continue

    print(f"Current subject: {sub}")
    sub_path = os.path.join(root, sub)

    if not os.path.isdir(sub_path):
        continue

    subject_id = int(sub.split("-")[-1])
    print(f"Subject ID: {subject_id}")

    # Iterate over sessions
    for ses in sorted(os.listdir(sub_path), key=natural_sort_key):
        if not ses.startswith("ses-"):
            continue

        print(f"  Current session: {ses}")
        eeg_path = os.path.join(sub_path, ses, "eeg")

        if not os.path.exists(eeg_path):
            print(f"❌ Skipping {sub}/{ses}: no eeg folder found.")
            continue

        set_files = {}
        events_files = {}

        for filename in os.listdir(eeg_path):
            filepath = os.path.join(eeg_path, filename)

            if filename.endswith("_eeg.bdf"):
                run_prefix = filename[:-len("_eeg.bdf")]
                set_files[run_prefix] = filepath

            elif filename.endswith("_events.tsv"):
                run_prefix = filename[:-len("_events.tsv")]
                events_files[run_prefix] = filepath

        all_run_prefixes = sorted(
            set(set_files.keys()) | set(events_files.keys()),
            key=natural_sort_key
        )

        # Only keep TASK_NAME runs
        all_run_prefixes = [
            run_prefix for run_prefix in all_run_prefixes
            if TASK_NAME in run_prefix
        ]
        
        if len(all_run_prefixes) == 0:
            print(f"❌ Skipping {sub}/{ses}: no run files found.")
            continue

        print(f"  Number of detected runs: {len(all_run_prefixes)}")

        for run_prefix in all_run_prefixes:
            set_file_path = set_files.get(run_prefix)
            events_file_path = events_files.get(run_prefix)

            print(f"    Current run: {run_prefix}")

            if set_file_path is None:
                print(f"    ❌ Missing .set file for {run_prefix}, skipping.")
                continue

            if events_file_path is None:
                print(f"    ❌ Missing events.tsv file for {run_prefix}, skipping.")
                continue

            print("  Start loading EEG data...")
            try:
                raw = mne.io.read_raw_bdf(
                    set_file_path,
                    preload=True,
                    verbose=False
                )
                raw = data_preprocessing(raw, common_channels, SAMPLE_RATE, notch_freq=50, l_freq=0.5, h_freq=40, verbose=True)
            except Exception as e:
                print(f"  ❌ Error loading or preprocessing {set_file_path}: {e}")
                continue
    
            print(f"Subject ID: {subject_id}")
            print("Start epoching and making X, y...")
    
            try:
                X, y = epoch_and_make_xy(
                    raw,
                    events_file_path,
                    tmin=TMIN,
                    tmax=TMAX,
                    baseline=BASELINE,
                    task_id=TASK_ID,
                    subject_id=subject_id,
                )
        
                # Save to memmap
                X_mm[cur:cur+X.shape[0]] = X
                y_mm[cur:cur+y.shape[0]] = y
                cur += X.shape[0]
            except Exception as e:
                print(f"  ❌ Error during epoching for {run_prefix}: {e}")
                continue
    
            print("------------------------------------------------\n")

        print("  ================================================\n")

print("DONE: cur =", cur, " expected N_total =", N_total)

# flush
del X_mm
del y_mm  

Current subject: sub-87957837
Subject ID: 87957837
  Current session: ses-1
  Number of detected runs: 1
    Current run: sub-87957837_ses-1_task-oddball
  Start loading EEG data...
Current channels:  ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'FC3', 'FCz', 'FC4', 'T7', 'C3', 'Cz', 'C4', 'T8', 'CP3', 'CPz', 'CP4', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'Oz', 'O2', 'VPVA', 'VNVB', 'HPHL', 'HNHR', 'Erbs', 'Mass', 'Status']
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
✔ Step 2: Picked common EEG channels (26): ['Fp1', 'Fp2', 'CP4', 'Pz', 'Cz', 'CP3', 'P8', 'P7', 'Fz', 'P3', 'FCz', 'Oz', 'C4', 'T8', 'FC3', 'F3', 'O2', 'F7', 'P4', 'O1', 'F8', 'FC4', 'CPz', 'T7', 'F4', 'C3']
✔ Step 1, Montage set: standard_1020.
✔ Step 3: Notch @ 50 Hz
✔ Step 4: Band-pass 0.5–40 Hz
ℹ Step 5: No bads to interpolate (set raw.info['bads'] first if needed)
✔ Step 6: Average reference

Fitting ICA to data using 26 channels (please be patient, this may take a while)
Selecting by n

## Create JSON file

In [38]:
meta = {
    "N": int(N_total),
    "T": SEG_LEN,
    "C": int(n_channels),
    "SAMPLE_RATE": SAMPLE_RATE,
    "TMIN": TMIN,
    "TMAX": TMAX,
    "BASELINE": BASELINE,
    "MONTAGE": MONTAGE,
    "CHANNELS": common_channels,
    "LABELS": {
        "disease_id": {-1: "set to unknown for now, wait for full ERP dataset release"},
        "stimulus_type": {0: "standard", 1: "target"},
        "subject_id": "integer from 1 to number of subjects",
        "task_id": {1: TASK_NAME},
    }
}
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved meta:", meta_path)

Saved meta: Processed\200Hz\TDBrain-ODD\meta.json


## Load and check the processed data

In [39]:
# load meta information
meta_path = meta_path  # if you already have meta_path in current notebook
with open(meta_path, "r") as f:
    meta = json.load(f)

N = meta["N"]
T = meta["T"]          # SEG_LEN
C = meta["C"]

print("Meta:")
for k, v in meta.items():
    print(f"{k}: {v}")
print("-------------------------------------")

# open memmap
X_mm = np.memmap(X_path, dtype='float32', mode='r', shape=(N, T, C))
y_mm = np.memmap(y_path, dtype='float32', mode='r', shape=(N, 4))

# subject_id is stored in y[:, 2]
subject_ids = np.unique(y_mm[:, 2]).astype(int)

total_trials = 0
for sid in sorted(subject_ids):
    # find all indices for this subject
    idx = np.where(y_mm[:, 2] == sid)[0]

    # compute shapes logically (do not load all X into memory)
    n_seg = len(idx)
    x_shape = (n_seg, T, C)    # logical X shape for this subject
    y_shape = (n_seg, 4)       # logical y shape for this subject
    total_trials += n_seg
    
    print(f"Subject ID {sid:03d}: X shape={x_shape}, y shape={y_shape}")

print("\nTotal subjects:", len(subject_ids))
print("Total trials:", total_trials)
# option: delete memmap views
del X_mm, y_mm

Meta:
N: 54400
T: 200
C: 26
SAMPLE_RATE: 200
TMIN: -0.2
TMAX: 0.8
BASELINE: [-0.2, 0.0]
MONTAGE: standard_1020
CHANNELS: ['Fp1', 'Fp2', 'CP4', 'Pz', 'Cz', 'CP3', 'P8', 'P7', 'Fz', 'P3', 'FCz', 'Oz', 'C4', 'T8', 'FC3', 'F3', 'O2', 'F7', 'P4', 'O1', 'F8', 'FC4', 'CPz', 'T7', 'F4', 'C3']
LABELS: {'disease_id': {'-1': 'set to unknown for now, wait for full ERP dataset release'}, 'stimulus_type': {'0': 'standard', '1': 'target'}, 'subject_id': 'integer from 1 to number of subjects', 'task_id': {'1': 'oddball'}}
-------------------------------------
Subject ID 87957840: X shape=(340, 200, 26), y shape=(340, 4)
Subject ID 87957968: X shape=(340, 200, 26), y shape=(340, 4)
Subject ID 87958016: X shape=(340, 200, 26), y shape=(340, 4)
Subject ID 87958056: X shape=(340, 200, 26), y shape=(340, 4)
Subject ID 87958064: X shape=(340, 200, 26), y shape=(340, 4)
Subject ID 87958104: X shape=(340, 200, 26), y shape=(340, 4)
Subject ID 87958152: X shape=(340, 200, 26), y shape=(340, 4)
Subject ID 87958